### POI dataset type check

In [13]:
import geopandas as gpd

gdf = gpd.read_file("6000POI.geojson")

# check type
geometry_counts = gdf.geometry.geom_type.value_counts()

print("Geometry type counts:")
print(geometry_counts)


Geometry type counts:
Point           5611
Polygon          568
MultiPolygon      22
LineString         7
Name: count, dtype: int64


In [1]:
import geopandas as gpd

gdf = gpd.read_file("limited_POI_20250702.geojson")

# check geometry type
geometry_counts = gdf.geometry.geom_type.value_counts()

print("Geometry type counts:")
print(geometry_counts)


Geometry type counts:
Point           2355
Polygon          568
MultiPolygon      22
LineString         7
Name: count, dtype: int64


### Modify all non-point records to type-point, using the centroid as its new coordinates

In [2]:
import geopandas as gpd
import json

# Load the original GeoJSON file
gdf = gpd.read_file("limited_POI_20250702.geojson")

# Drop records with missing geometry
gdf = gdf[~gdf["geometry"].isna()]

# Replace non-Point geometries with their centroids
gdf["geometry"] = gdf["geometry"].apply(
    lambda geom: geom if geom.geom_type == "Point" else geom.centroid
)

# Save to a temporary file
temp_file = "temp_output.geojson"
gdf.to_file(temp_file, driver="GeoJSON")

# Open the temporary file and update the name field in the FeatureCollection
with open(temp_file, "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

geojson_data["name"] = "limited_POI_20250702_new"

# Write the final output file
output_file = "limited_POI_20250702_new.geojson"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(geojson_data, f, ensure_ascii=False, indent=2)

In [3]:
# Load the GeoJSON file
with open('limited_POI_20250702_new.geojson', 'r', encoding='utf-8') as f:
    data = json.load(f)

target_tag = 'shop'

# Loop through features and print those that contain the target tag
for i, feature in enumerate(data['features']):
    props = feature.get('properties', {})
    if target_tag in props:
        print(f"Feature {i} contains '{target_tag}' = {props[target_tag]}")

Feature 0 contains 'shop' = None
Feature 1 contains 'shop' = None
Feature 2 contains 'shop' = None
Feature 3 contains 'shop' = None
Feature 4 contains 'shop' = None
Feature 5 contains 'shop' = None
Feature 6 contains 'shop' = None
Feature 7 contains 'shop' = None
Feature 8 contains 'shop' = None
Feature 9 contains 'shop' = None
Feature 10 contains 'shop' = None
Feature 11 contains 'shop' = None
Feature 12 contains 'shop' = None
Feature 13 contains 'shop' = None
Feature 14 contains 'shop' = None
Feature 15 contains 'shop' = None
Feature 16 contains 'shop' = None
Feature 17 contains 'shop' = None
Feature 18 contains 'shop' = None
Feature 19 contains 'shop' = None
Feature 20 contains 'shop' = None
Feature 21 contains 'shop' = None
Feature 22 contains 'shop' = None
Feature 23 contains 'shop' = None
Feature 24 contains 'shop' = None
Feature 25 contains 'shop' = None
Feature 26 contains 'shop' = None
Feature 27 contains 'shop' = None
Feature 28 contains 'shop' = None
Feature 29 contains 'sho

In [4]:
import json
from collections import defaultdict
import pandas as pd

# Load the GeoJSON file
with open('limited_POI_20250702_new.geojson', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Common tags used to describe place types
type_tags = ['shop', 'amenity', 'leisure', 'tourism', 'building', 'craft']

# Count the occurrences of each type across all features
type_counts = defaultdict(int)

for feature in data['features']:
    props = feature.get('properties', {})
    for tag in type_tags:
        value = props.get(tag)
        if value not in [None, '', 'unknown']:
            type_counts[value] += 1

# Convert to DataFrame and sort by count
df = pd.DataFrame(type_counts.items(), columns=['type', 'count'])
df_sorted = df.sort_values(by='count', ascending=False)

# Display the top 20 most frequent types
print(df_sorted.head(20))

                type  count
20              cafe    692
39        restaurant    674
13               bar    524
4               park    351
34           gallery    224
44           toilets    188
3                yes    128
1         attraction     98
37            museum     91
0                atm     58
43  department_store     55
42              mall      7
22            bakery      5
8               ship      4
26           caterer      4
40            retail      4
16           alcohol      4
11        commercial      4
32            pastry      3
18             dance      2


In [5]:
with open('limited_POI_20250702_new.geojson', 'r', encoding='utf-8') as f:
    data = json.load(f)

museum_features = []

# Tags commonly used to classify place types
type_tags = ['shop', 'amenity', 'leisure', 'tourism', 'building', 'craft']

# Collect features that are classified as 'museum'
for feature in data['features']:
    props = feature.get('properties', {})
    for tag in type_tags:
        if props.get(tag) == 'museum':
            museum_features.append(feature)
            break  # Stop at the first matching tag

# Build a new GeoJSON with only museum features
museum_geojson = {
    "type": "FeatureCollection",
    "features": museum_features
}

# Save to a new file
with open('museums_only.geojson', 'w', encoding='utf-8') as f:
    json.dump(museum_geojson, f, ensure_ascii=False, indent=2)

print(f"Extracted {len(museum_features)} museum features.")

Extracted 86 museum features.


In [6]:
with open('museums_only.geojson', 'r', encoding='utf-8') as f:
    data = json.load(f)

no_name_count = 0
total_count = len(data['features'])

# Count how many museum features are missing a name
for feature in data['features']:
    props = feature.get('properties', {})
    name = props.get('name')
    if not name or str(name).strip() == '':
        no_name_count += 1

print(f"Total museum features: {total_count}")
print(f"Museums without a name: {no_name_count}")
print(f"Percentage without name: {no_name_count / total_count * 100:.2f}%")

Total museum features: 86
Museums without a name: 0
Percentage without name: 0.00%


In [7]:
from shapely.geometry import shape, Point

with open('limited_POI_20250702_new.geojson', 'r', encoding='utf-8') as f:
    base_data = json.load(f)

# Load additional museum data
with open('add_Meu.geojson', 'r', encoding='utf-8') as f:
    new_data = json.load(f)

# Generate a unique key based on name and coordinates
def feature_key(feature):
    props = feature.get('properties', {})
    name_raw = props.get('name')
    name = str(name_raw).strip().lower() if isinstance(name_raw, str) else None

    geom = shape(feature['geometry'])
    if not geom.is_empty:
        centroid = geom.centroid
        coords = tuple(round(c, 6) for c in centroid.coords[0])
    else:
        coords = None

    return (name, coords) if name else ('__no_name__', coords)

existing_keys = set()
cleaned_base_features = []

for feature in base_data['features']:
    key = feature_key(feature)
    existing_keys.add(key)
    cleaned_base_features.append(feature)

cleaned_add_features = []

for feature in new_data['features']:
    props = feature.get('properties', {})
    name = props.get('name', '').strip().lower()
    
    geom = shape(feature['geometry'])
    if geom.geom_type != 'Point':
        centroid = geom.centroid
        feature['geometry'] = {
            "type": "Point",
            "coordinates": [centroid.x, centroid.y]
        }
        geom = centroid
    
    coords = tuple(round(c, 6) for c in geom.coords[0])
    key = (name, coords)

    if key not in existing_keys:
        existing_keys.add(key)
        cleaned_add_features.append(feature)

merged_features = cleaned_base_features + cleaned_add_features

merged_geojson = {
    "type": "FeatureCollection",
    "features": merged_features
}

with open('limited_POI_merged.geojson', 'w', encoding='utf-8') as f:
    json.dump(merged_geojson, f, ensure_ascii=False, indent=2)

print(f"Merge completed. Added museums: {len(cleaned_add_features)}, total records: {len(merged_features)}")

Merge completed. Added museums: 95, total records: 3047


### Coordinate correction

In [3]:
gdf = gpd.read_file('limited_POI_merged.geojson')

# Define target POI types
target_types = [
    'cafe', 'restaurant', 'bar', 'park', 'gallery', 'museum',
    'toilets', 'attraction', 'atm', 'department_store', 'mall', 'bakery'
]

# Filter by selected types
filtered = gdf[gdf['type'].isin(target_types)]

# Keep relevant columns
result = filtered[['type', 'name']]

# Print names grouped by type
for t, group in result.groupby('type'):
    print(f"\n=== {t.upper()} ===")
    print(group['name'].dropna().tolist())


=== ATM ===
['Bank of America', 'Bank of America', 'Chase', 'Citibank', 'Bank of America', 'Citibank', 'Wells Fargo', 'Bank of America', 'Bank of America', 'Chase', 'Bank of America', 'Capital One', 'Chase', 'Chase', 'Chase', 'Chase', 'Chase', 'Chase', 'Capital One', 'Bank of America', 'Bank of America', 'Capital One', 'Citibank', 'Chase', 'Chase', 'Bank of America', 'Bank of America', 'Citibank', 'Chase', 'Chase', 'Citibank', 'Bank of America', 'Citibank', 'Chase', 'Citibank', 'Chase', 'Bank of America', 'Bank of America', 'UNFCU', 'Santander', 'PNC Bank', 'PNC Bank', 'Bank of America', 'Bank of America ATM', 'Citibank', 'Bank of America', 'Bank of America', 'Bank of America', 'Bank of America', 'Bank of America', 'Bank of America', 'Bank of America', 'Santander', 'Citibank', 'Bank of America', 'TD Bank', 'Bank of America', 'Citibank']

=== ATTRACTION ===
['The Dakota', 'The Cloisters', 'The High Line', 'Manhattan Bridge', 'Rockefeller Center', 'Empire State Building', 'Blockhouse No

In [9]:
import requests
import pandas as pd
import time
from urllib.parse import quote

API_KEY = "0d64bdf904cb40859c631f2562d4e41a"

# Load deduplicated gallery names
galleries = pd.read_csv("galleries_unique.csv")
results = []

for index, row in galleries.iterrows():
    query = f"{row['name']}, New York, NY"
    encoded_query = quote(query)
    url = f"https://api.geoapify.com/v1/geocode/search?text={encoded_query}&apiKey={API_KEY}"

    headers = {"Accept": "application/json"}

    try:
        resp = requests.get(url, headers=headers)
        if resp.status_code == 200:
            data = resp.json()
            if data["features"]:
                feature = data["features"][0]
                coords = feature["geometry"]["coordinates"]
                matched_address = feature["properties"].get("formatted", "Unknown address")
                results.append({
                    "name": row["name"],
                    "latitude": coords[1],
                    "longitude": coords[0],
                    "matched_address": matched_address
                })
                print(f"Matched: {row['name']} -> ({coords[1]}, {coords[0]})")
            else:
                print(f"No result: {row['name']}")
        else:
            print(f"Request failed: {row['name']} (status code {resp.status_code})")
    except Exception as e:
        print(f"Error with {row['name']}: {e}")

    time.sleep(1)

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv("gallery_coords.csv", index=False)
print("Saved to gallery_coords.csv")

Matched: 303 Gallery -> (40.7475429, -74.0075391)
Matched: 532 Gallery Thomas Jaeckel -> (40.7495009, -74.0047674)
Matched: 81 Leonard Gallery -> (40.7174539, -74.0049014)
Matched: ABRI MARS -> (40.7202681, -73.9851754)
Matched: ACA Galleries -> (40.7465429, -74.0068474)
Matched: Amanita -> (40.7251309, -73.9919779)
Matched: Alexander Gray Associates Gallery -> (40.7497859, -74.0036422)
Matched: Alexander and Bonin -> (40.7190232, -74.0036388)
Matched: All Street Gallery -> (40.725038, -73.9883259)
Matched: Allouche Gallery -> (40.7227411, -74.0003875)
Matched: Anat Egbi -> (40.7176003, -74.00295089140083)
Matched: Andrew Kreps Gallery -> (40.7181761, -74.0020133)
Matched: Anna Zorina Gallery -> (40.7489431, -74.0053524)
Matched: Anya & Andrew Shiva Gallery -> (40.771366, -73.9902725)
Matched: Art Clinic -> (40.7187633, -73.9831736)
Matched: Art Projects International -> (40.7226366, -74.0098159)
Matched: Art+Ray -> (40.7290786, -73.9867283)
Matched: Arsenal Contemporary Art -> (40.721

In [10]:
import requests
import pandas as pd
import time
from urllib.parse import quote
import csv

API_KEY = "0d64bdf904cb40859c631f2562d4e41a"

# Load deduplicated museum names with proper encoding and quoting
try:
    museums = pd.read_csv("museums_unique.csv", encoding="utf-8", quoting=csv.QUOTE_ALL)
except Exception as e:
    print(f"Failed to load CSV: {e}")
    exit()

results = []

for index, row in museums.iterrows():
    query = f"{row['name']}, New York, NY"
    encoded_query = quote(query)
    url = f"https://api.geoapify.com/v1/geocode/search?text={encoded_query}&apiKey={API_KEY}"
    headers = {"Accept": "application/json"}

    try:
        resp = requests.get(url, headers=headers)
        if resp.status_code == 200:
            data = resp.json()
            if data["features"]:
                feature = data["features"][0]
                coords = feature["geometry"]["coordinates"]
                matched_address = feature["properties"].get("formatted", "Unknown address")
                results.append({
                    "name": row["name"],
                    "latitude": coords[1],
                    "longitude": coords[0],
                    "matched_address": matched_address
                })
                print(f"Matched: {row['name']} -> ({coords[1]}, {coords[0]})")
            else:
                print(f"No result: {row['name']}")
        else:
            print(f"Request failed for {row['name']} (status {resp.status_code})")
    except Exception as e:
        print(f"Error fetching {row['name']}: {e}")

    time.sleep(1)

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv("museum_coords.csv", index=False, encoding="utf-8")
print("Saved to museum_coords.csv")

Matched: 9/11 Memorial & Museum -> (40.711452699999995, -74.01267026823709)
Matched: American Folk Art Museum -> (40.7732296, -73.9815958)
Matched: American Museum of Natural History -> (40.78110065, -73.97423619833333)
Matched: Asia Society -> (40.76983905, -73.9642912633345)
Matched: Banksy Museum -> (40.7194141, -74.001447)
Matched: Center for Architecture -> (40.72873, -73.9985805)
Matched: Children's Cultural Center of Native America -> (40.8321455, -73.9448716)
Matched: Children's Museum of Manhattan -> (40.7858779, -73.97726088274193)
Matched: Commanding Officer's Quarters -> (40.69036335, -74.01314010274635)
Matched: Cooper–Hewitt,  Smithsonian Design Museum -> (40.7127281, -74.0060152)
Matched: Dream House -> (40.7184994, -74.0048345)
Matched: Dyckman Farmhouse Museum -> (40.8673822, -73.9228859)
Matched: El Museo Del Barrio -> (40.7931208, -73.95133506373483)
Matched: Fraunces Tavern Museum -> (40.703387, -74.011354)
Matched: Frick Collection -> (40.77125355, -73.967096121362

In [11]:
import requests
import pandas as pd
import time
from urllib.parse import quote
import csv

API_KEY = "0d64bdf904cb40859c631f2562d4e41a"

# Check the first few lines of the CSV file for debugging
print("Previewing CSV content:")
with open("parks_unique.csv", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if i <= 12:
            print(f"Line {i}: {line.strip()}")
        if i == 12:
            break

# Load deduplicated park list
try:
    parks = pd.read_csv("parks_unique.csv", encoding="utf-8", quoting=csv.QUOTE_ALL)
    print(f"Loaded {len(parks)} records from CSV")
except Exception as e:
    print(f"Failed to load CSV: {e}")
    exit()

results = []

for index, row in parks.iterrows():
    query = f"{row['name']}, New York, NY"
    encoded_query = quote(query)
    url = f"https://api.geoapify.com/v1/geocode/search?text={encoded_query}&apiKey={API_KEY}"
    headers = {"Accept": "application/json"}

    try:
        resp = requests.get(url, headers=headers)
        if resp.status_code == 200:
            data = resp.json()
            if data["features"]:
                feature = data["features"][0]
                coords = feature["geometry"]["coordinates"]
                matched_address = feature["properties"].get("formatted", "Unknown address")
                results.append({
                    "name": row["name"],
                    "latitude": coords[1],
                    "longitude": coords[0],
                    "matched_address": matched_address
                })
                print(f"Matched: {row['name']} -> ({coords[1]}, {coords[0]})")
            else:
                print(f"No match found for: {row['name']}")
        else:
            print(f"Request failed for {row['name']} (status {resp.status_code})")
    except Exception as e:
        print(f"Error while requesting {row['name']}: {e}")

    time.sleep(1)

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv("park_coords.csv", index=False, encoding="utf-8")
print("Results saved to park_coords.csv")

Previewing CSV content:
Line 1: name
Line 2: "103rd Street Community Garden"
Line 3: "132nd Street Block Association Park"
Line 4: "148th Street Block Association Park"
Line 5: "A Philip Randolph Square"
Line 6: Abe Lebewohl Park
Line 7: Abingdon Square
Line 8: African Burial Ground National Monument
Line 9: Ahearn Park
Line 10: Albert Capsouto Park
Line 11: Alexander Hamilton Playground
Line 12: Anibal Aviles Playground
Loaded 298 records from CSV
Matched: 103rd Street Community Garden -> (40.79092815, -73.94861155036415)
Matched: 132nd Street Block Association Park -> (40.81232, -73.94315290642419)
Matched: 148th Street Block Association Park -> (40.8243789, -73.93899153382813)
Matched: A Philip Randolph Square -> (40.80377155, -73.95246473306219)
Matched: Abe Lebewohl Park -> (40.72999515, -73.98697675108093)
Matched: Abingdon Square -> (40.73731825, -74.0054416110881)
Matched: African Burial Ground National Monument -> (40.71452515, -74.0044656547183)
Matched: Ahearn Park -> (40.71

In [12]:
import requests
import pandas as pd
import time
from urllib.parse import quote
import csv

API_KEY = "0d64bdf904cb40859c631f2562d4e41a"

# Check the first few lines of the CSV file for debugging
print("Previewing CSV content:")
with open("restaurants_unique.csv", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if i <= 12:
            print(f"Line {i}: {line.strip()}")
        if i == 12:
            break

# Load deduplicated restaurant list
try:
    restaurants = pd.read_csv("restaurants_unique.csv", encoding="utf-8", quoting=csv.QUOTE_ALL)
    print(f"Loaded {len(restaurants)} records from CSV")
except Exception as e:
    print(f"Failed to load CSV: {e}")
    exit()

results = []

for index, row in restaurants.iterrows():
    query = f"{row['name']}, New York, NY"
    encoded_query = quote(query)
    url = f"https://api.geoapify.com/v1/geocode/search?text={encoded_query}&apiKey={API_KEY}"
    headers = {"Accept": "application/json"}

    try:
        resp = requests.get(url, headers=headers)
        if resp.status_code == 200:
            data = resp.json()
            if data["features"]:
                feature = data["features"][0]
                coords = feature["geometry"]["coordinates"]
                matched_address = feature["properties"].get("formatted", "Unknown address")
                results.append({
                    "name": row["name"],
                    "latitude": coords[1],
                    "longitude": coords[0],
                    "matched_address": matched_address
                })
                print(f"Matched: {row['name']} -> ({coords[1]}, {coords[0]})")
            else:
                print(f"No match found for: {row['name']}")
        else:
            print(f"Request failed for {row['name']} (status {resp.status_code})")
    except Exception as e:
        print(f"Error while requesting {row['name']}: {e}")

    time.sleep(1)

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv("restaurant_coords.csv", index=False, encoding="utf-8")
print("Results saved to restaurant_coords.csv")

Previewing CSV content:
Line 1: name
Line 2: "181 Cabrini"
Line 3: "$1.50 Fresh Pizza"
Line 4: "& Son Steakeasy"
Line 5: "19 Cleveland"
Line 6: "44 & X"
Line 7: "5ive Spice"
Line 8: "886"
Line 9: A Taste of Seafood
Line 10: Abace Sushi
Line 11: Absolute Greek (Food Truck)
Line 12: Abyssinia
Loaded 545 records from CSV
Matched: 181 Cabrini -> (40.8538917, -73.939247)
Matched: $1.50 Fresh Pizza -> (40.7646117, -73.9824665)
Matched: & Son Steakeasy -> (40.7338544, -73.9987056)
Matched: 19 Cleveland -> (40.7216382, -73.9972123)
Matched: 44 & X -> (40.7609658, -73.994264)
Matched: 5ive Spice -> (40.7408703, -73.9814116)
Matched: 886 -> (40.7127281, -74.0060152)
Matched: A Taste of Seafood -> (40.7934257, -73.943551)
Matched: Abace Sushi -> (40.7621602, -73.9901271)
Matched: Absolute Greek (Food Truck) -> (40.703713, -74.0087506)
Matched: Abyssinia -> (40.8160376, -73.9458914)
Matched: Agaveria La Diagonal -> (40.8060758, -73.9530327)
Matched: Ajisen Ramen -> (40.7466178, -73.9921722)
Matche

In [13]:
import requests
import pandas as pd
import time
from urllib.parse import quote
import csv

API_KEY = "0d64bdf904cb40859c631f2562d4e41a"

# Preview the CSV content
print("Previewing CSV content:")
with open("attractions_unique.csv", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if i <= 12:
            print(f"Line {i}: {line.strip()}")
        if i == 12:
            break

# Load deduplicated list of attractions
try:
    attractions = pd.read_csv("attractions_unique.csv", encoding="utf-8", quoting=csv.QUOTE_ALL)
    print(f"Loaded {len(attractions)} records from CSV")
except Exception as e:
    print(f"Failed to load CSV: {e}")
    exit()

results = []

# Query Geoapify for each attraction name
for index, row in attractions.iterrows():
    query = f"{row['name']}, New York, NY"
    encoded_query = quote(query)
    url = f"https://api.geoapify.com/v1/geocode/search?text={encoded_query}&apiKey={API_KEY}"
    headers = {"Accept": "application/json"}

    try:
        resp = requests.get(url, headers=headers)
        if resp.status_code == 200:
            data = resp.json()
            if data["features"]:
                feature = data["features"][0]
                coords = feature["geometry"]["coordinates"]
                matched_address = feature["properties"].get("formatted", "Unknown address")
                results.append({
                    "name": row["name"],
                    "latitude": coords[1],
                    "longitude": coords[0],
                    "matched_address": matched_address
                })
                print(f"Matched: {row['name']} -> ({coords[1]}, {coords[0]})")
            else:
                print(f"No match found for: {row['name']}")
        else:
            print(f"Request failed for {row['name']} (status {resp.status_code})")
    except Exception as e:
        print(f"Error while requesting {row['name']}: {e}")

    time.sleep(1)

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv("attraction_coords.csv", index=False, encoding="utf-8")
print("Results saved to attraction_coords.csv")

Previewing CSV content:
Line 1: name
Line 2: Ambrose
Line 3: Anton Kern Gallery
Line 4: Archibald’s Townhouse
Line 5: "Barthman's Sidewalk Clock"
Line 6: Blackwell House
Line 7: Blockhouse No. 1
Line 8: Bloody Angle
Line 9: Bow Bridge
Line 10: Brooklyn Bridge
Line 11: Central Park Carousel
Line 12: Chelsea Flea Market
Loaded 86 records from CSV
Matched: Ambrose -> (40.705297, -74.00242934457971)
Matched: Anton Kern Gallery -> (40.7611131, -73.9739449)
Matched: Archibald’s Townhouse -> (40.773329, -73.9656246)
Matched: Barthman's Sidewalk Clock -> (40.7098195, -74.0099251)
Matched: Blackwell House -> (40.7603285, -73.95112829495261)
Matched: Blockhouse No. 1 -> (40.7986699, -73.9562876)
Matched: Bloody Angle -> (40.7144127, -73.9981498)
Matched: Bow Bridge -> (40.77576035, -73.97176891989135)
Matched: Brooklyn Bridge -> (40.705749811166456, -73.99627645923147)
Matched: Central Park Carousel -> (40.76994265, -73.97524754999999)
Matched: Chelsea Flea Market -> (40.7436928, -73.9903215)
Ma

In [10]:
gdf = gpd.read_file("limited_POI_merged.geojson")

# Filter features where shop type is department_store or mall
filtered = gdf[gdf['shop'].isin(['department_store', 'mall'])]

# Drop rows with missing names
filtered = filtered.dropna(subset=['name'])

count = 0

# Print type and name for each matching record
for _, row in filtered.iterrows():
    print(f"{row['shop']}: {row['name']}")
    count += 1

print(count)

mall: Westfield World Trade Center
department_store: Saks Fifth Avenue
department_store: Bloomingdale's
mall: Manhattan Mall
department_store: Dover Street Market
department_store: Macy's
department_store: TJ Maxx
department_store: Bloomingdale's
department_store: Easy Shopping
mall: East River Plaza
department_store: Target
mall: East Broadway Mall
mall: Chelsea Market
department_store: TJ Maxx
department_store: Marshalls
department_store: Target
department_store: Barneys New York
department_store: Burlington
department_store: Target
department_store: Pearl River Mart
department_store: Target
department_store: Kartell
department_store: Bergdorf Goodman, Canada Goose
department_store: Target
department_store: Marshalls
department_store: Target
department_store: Marshalls
department_store: Bergdorf Goodman
department_store: Bloomingdale's
department_store: Muji
department_store: TJ Maxx
department_store: TJ Maxx
mall: The Shops at Columbus Circle
department_store: Burlington
department_

In [18]:
import requests
import pandas as pd
import time
from urllib.parse import quote
import csv

# Geoapify API key
API_KEY = "0d64bdf904cb40859c631f2562d4e41a"  # Replace with your actual key

# Preview the first few lines of the CSV
print("Previewing CSV content:")
with open("malls_unique.csv", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if i <= 12:
            print(f"Line {i}: {line.strip()}")
        if i == 12:
            break

# Load mall list from CSV
try:
    attractions = pd.read_csv("malls_unique.csv", encoding="utf-8", quoting=csv.QUOTE_ALL)
    print(f"CSV loaded successfully. {len(attractions)} records found.")
except Exception as e:
    print(f"Failed to read CSV: {e}")
    exit()

results = []

# Geocode each mall using Geoapify
for index, row in attractions.iterrows():
    query = f"{row['name']}, New York, NY"
    encoded_query = quote(query)
    url = f"https://api.geoapify.com/v1/geocode/search?text={encoded_query}&apiKey={API_KEY}"
    headers = {"Accept": "application/json"}

    try:
        resp = requests.get(url, headers=headers)
        if resp.status_code == 200:
            data = resp.json()
            if data["features"]:
                feature = data["features"][0]
                coords = feature["geometry"]["coordinates"]
                matched_address = feature["properties"].get("formatted", "Unknown address")
                results.append({
                    "name": row["name"],
                    "latitude": coords[1],
                    "longitude": coords[0],
                    "matched_address": matched_address
                })
                print(f"Matched: {row['name']} -> ({coords[1]}, {coords[0]})")
            else:
                print(f"No result for: {row['name']}")
        else:
            print(f"Error {resp.status_code} for: {row['name']}")
    except Exception as e:
        print(f"Request failed for {row['name']}: {e}")

    time.sleep(1)

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv("malls_coords.csv", index=False, encoding="utf-8")
print("Results saved to malls_coords.csv")

Previewing CSV content:
Line 1: name
Line 2: Chelsea Market
Line 3: East Broadway Mall
Line 4: East River Plaza
Line 5: Koreatown Shopping Court
Line 6: Manhattan Mall
Line 7: The Shops at Columbus Circle
Line 8: Westfield World Trade Center
CSV loaded successfully. 7 records found.
Matched: Chelsea Market -> (40.7420513, -74.0048973)
Matched: East Broadway Mall -> (40.71400895, -73.99439502318911)
Matched: East River Plaza -> (40.799648127216194, -73.84322940128739)
Matched: Koreatown Shopping Court -> (40.7471497, -73.9858691)
Matched: Manhattan Mall -> (40.74915515, -73.98928999240688)
Matched: The Shops at Columbus Circle -> (40.7684026, -73.9828296)
Matched: Westfield World Trade Center -> (40.71093215, -74.01164249076538)
Results saved to malls_coords.csv


In [19]:
import requests
import pandas as pd
import time
from urllib.parse import quote
import csv

# Geoapify API key
API_KEY = "0d64bdf904cb40859c631f2562d4e41a"  # Replace with your actual key

# Preview the CSV content
print("Previewing CSV content:")
with open("department_stores_unique.csv", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if i <= 12:
            print(f"Line {i}: {line.strip()}")
        if i == 12:
            break

# Load department store names
try:
    attractions = pd.read_csv("department_stores_unique.csv", encoding="utf-8", quoting=csv.QUOTE_ALL)
    print(f"CSV loaded successfully. {len(attractions)} rows.")
except Exception as e:
    print(f"Failed to read CSV: {e}")
    exit()

results = []

# Query Geoapify for each name
for index, row in attractions.iterrows():
    query = f"{row['name']}, New York, NY"
    encoded_query = quote(query)
    url = f"https://api.geoapify.com/v1/geocode/search?text={encoded_query}&apiKey={API_KEY}"
    headers = {"Accept": "application/json"}

    try:
        resp = requests.get(url, headers=headers)
        if resp.status_code == 200:
            data = resp.json()
            if data["features"]:
                feature = data["features"][0]
                coords = feature["geometry"]["coordinates"]
                matched_address = feature["properties"].get("formatted", "Unknown address")
                results.append({
                    "name": row["name"],
                    "latitude": coords[1],
                    "longitude": coords[0],
                    "matched_address": matched_address
                })
                print(f"Matched: {row['name']} -> ({coords[1]}, {coords[0]})")
            else:
                print(f"No result for: {row['name']}")
        else:
            print(f"Error {resp.status_code} for: {row['name']}")
    except Exception as e:
        print(f"Request failed for {row['name']}: {e}")

    time.sleep(1)

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv("department_stores_coords.csv", index=False, encoding="utf-8")
print("Results saved to department_stores_coords.csv")

Previewing CSV content:
Line 1: name
Line 2: "Barneys New York"
Line 3: Bergdorf Goodman
Line 4: "Bergdorf Goodman, Canada Goose"
Line 5: Bloomingdale's
Line 6: Burlington
Line 7: "classic watch inc suite 17"
Line 8: Dover Street Market
Line 9: Easy Shopping
Line 10: Kartell
Line 11: Macy's
Line 12: "Marshalls"
CSV loaded successfully. 21 rows.
Matched: Barneys New York -> (40.7647513, -73.9709644)
Matched: Bergdorf Goodman -> (40.7633245, -73.9733602)
Matched: Bergdorf Goodman, Canada Goose -> (40.7634607, -73.9739286)
Matched: Bloomingdale's -> (40.762210350000004, -73.96717434559059)
Matched: Burlington -> (40.6838452, -73.9754366)
Matched: classic watch inc suite 17 -> (40.7572519, -73.9802443)
Matched: Dover Street Market -> (40.744105700000006, -73.98171685176533)
Matched: Easy Shopping -> (40.7906331, -73.94573769086446)
Matched: Kartell -> (40.7219268, -74.0020843)
Matched: Macy's -> (40.75088945, -73.98925506526803)
Matched: Marshalls -> (40.590566949999996, -73.95342858110324

#### Supplementary store information

In [20]:
import requests
import pandas as pd
import time

API_KEY = "0d64bdf904cb40859c631f2562d4e41a"  

# Manhattan bounding box
MANHATTAN_BBOX = "rect:-74.0200,40.7000,-73.9200,40.8700"

# Max records to fetch
MAX_RECORDS = 300

results = []
offset = 0

while len(results) < MAX_RECORDS:
    url = f"https://api.geoapify.com/v2/places?categories=commercial.shopping_mall&filter={MANHATTAN_BBOX}&limit=100&offset={offset}&apiKey={API_KEY}"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        features = data.get("features", [])
        if not features:
            break
        for feature in features:
            props = feature["properties"]
            results.append({
                "name": props.get("name", "Unknown"),
                "latitude": props["lat"],
                "longitude": props["lon"],
                "matched_address": props.get("formatted", "Unknown")
            })
            if len(results) >= MAX_RECORDS:
                break
        offset += 100
        print(f"Fetched {len(features)} records. Total so far: {len(results)}")
    else:
        print(f"Error {response.status_code}: {response.text}")
        break

    time.sleep(1)  # Respect rate limits

# Save to CSV
df = pd.DataFrame(results)
df.to_csv("manhattan_shopping_mall.csv", index=False, encoding="utf-8")
print(f"Saved to manhattan_shopping_mall.csv with {len(results)} records.")

Fetched 15 records. Total so far: 15
Saved to manhattan_shopping_mall.csv with 15 records.


In [21]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Input CSV files and corresponding tag mappings
file_info = {
    'manhattan_department_stores.csv': {'field': 'shop', 'value': 'department_store', 'category': 'shop'},
    'manhattan_shopping_mall.csv':     {'field': 'shop', 'value': 'mall', 'category': 'shop'},
    'restaurant_coords.csv':           {'field': 'amenity', 'value': 'restaurant', 'category': 'amenity'},
    'park_coords.csv':                 {'field': 'leisure', 'value': 'park', 'category': 'leisure'},
    'museum_coords.csv':               {'field': 'tourism', 'value': 'museum', 'category': 'tourism'},
    'gallery_coords.csv':              {'field': 'tourism', 'value': 'gallery', 'category': 'tourism'},
}

all_data = []
counter = 1  # ID counter

def read_csv_with_fallback(file):
    for enc in ['utf-8-sig', 'utf-8', 'ISO-8859-1', 'gbk']:
        try:
            return pd.read_csv(file, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f"Cannot read file: {file}. Please check its encoding.")

for filename, mapping in file_info.items():
    df = read_csv_with_fallback(filename)

    if 'matched_address' in df.columns:
        df = df.rename(columns={'matched_address': 'address'})

    df['category'] = mapping['category']
    for key in ['shop', 'amenity', 'leisure', 'tourism']:
        df[key] = None
    df[mapping['field']] = mapping['value']

    num_rows = len(df)
    df['id'] = [f'poi_{str(i).zfill(5)}' for i in range(counter, counter + num_rows)]
    counter += num_rows

    all_data.append(df)

merged_df = pd.concat(all_data, ignore_index=True)

geometry = [Point(xy) for xy in zip(merged_df['longitude'], merged_df['latitude'])]
gdf = gpd.GeoDataFrame(merged_df, geometry=geometry, crs="EPSG:4326")

gdf.to_file("merged_POI.geojson", driver="GeoJSON")

print("Saved merged_POI.geojson with POI IDs, categories, and addresses.")

Saved merged_POI.geojson with POI IDs, categories, and addresses.


In [17]:
import geopandas as gpd

# Load the GeoJSON file
gdf = gpd.read_file("merged_POI.geojson")

# Standardize tag values to lowercase strings
for col in ['tourism', 'leisure', 'amenity', 'shop']:
    if col in gdf.columns:
        gdf[col] = gdf[col].astype(str).str.lower()

# Define masks for each POI type
gallery_mask = gdf['tourism'] == 'gallery'
museum_mask = gdf['tourism'] == 'museum'
park_mask = gdf['leisure'] == 'park'
restaurant_mask = gdf['amenity'] == 'restaurant'
shop_mask = gdf['shop'].isin([
    'clothes', 'supermarket', 'department_store', 'mall', 'convenience'
])

# Count number of records in each category
counts = {
    'gallery': gallery_mask.sum(),
    'museum': museum_mask.sum(),
    'park': park_mask.sum(),
    'restaurant': restaurant_mask.sum(),
    'shop': shop_mask.sum()
}

# Print results
print("POI counts by category:")
for category, count in counts.items():
    print(f"{category}: {count}")


POI counts by category:
gallery: 198
museum: 84
park: 298
restaurant: 545
shop: 85
